In [10]:
import sys
sys.path.append("../src")

import pandas as pd
import yfinance as yf
import metrics

Choice of asset universe. 

Need large variety uncorrelated assets from different countries. 

SPY : snp500
QQQ : us tech
EWU : big entreprise UK
EZU : actions europe
GLD : gold
VNQ : real estate
EEM : emerging markets (china, inde, etc)
IEF : us 7-10 year bonds
SHY : us 3 year bonds



In [11]:
import yfinance as yf
tickers = ["SPY", "QQQ", "EWU", "EZU", "GLD", "VNQ", "EEM", "IEF", "SHY"]
for t in tickers:
    data = yf.download(t, start="1990-01-01", progress=False)
    if len(data) > 0:
        print(f"{t}: from {data.index[0].date()}, {len(data)} lines")
    else:
        print(f"{t}: NO DATA")

SPY: from 1993-01-29, 8469 lines
QQQ: from 1999-03-10, 6927 lines
EWU: from 1996-03-18, 7678 lines
EZU: from 2000-07-31, 6575 lines
GLD: from 2004-11-18, 5494 lines
VNQ: from 2004-09-29, 5530 lines
EEM: from 2003-04-14, 5898 lines
IEF: from 2002-07-30, 6076 lines
SHY: from 2002-07-30, 6076 lines


Maintenant que jai choisi et que je connais les dates de debut, je vais telecharger

In [12]:
import yfinance as yf
import pandas as pd

tickers = ["SPY", "QQQ", "EWU", "EZU", "GLD", "VNQ", "EEM", "IEF", "SHY"]
data = yf.download(tickers, start="2005-01-01", progress=False)
#print(data.columns)
prices = data["Close"]
print(prices.head())
print(prices.shape)
print(prices.isna().sum()) #il y a til des NAN

prices.to_csv("../data/etf_prices_raw.csv")

Ticker            EEM        EWU        EZU        GLD        IEF        QQQ  \
Date                                                                           
2005-01-03  14.487372  15.935783  19.757853  43.020000  47.438812  33.581028   
2005-01-04  14.041328  15.643306  19.457207  42.740002  47.143444  32.968899   
2005-01-05  13.869439  15.652167  19.336946  42.669998  47.221500  32.764877   
2005-01-06  13.860010  15.643306  19.391609  42.150002  47.266068  32.603325   
2005-01-07  13.886850  15.661029  19.175695  41.840000  47.227062  32.773357   

Ticker            SHY        SPY        VNQ  
Date                                         
2005-01-03  53.612900  80.973488  22.350878  
2005-01-04  53.494404  79.984077  22.014961  
2005-01-05  53.507610  79.432129  21.283125  
2005-01-06  53.533932  79.836021  21.447086  
2005-01-07  53.540508  79.721573  21.399105  
(5464, 9)
Ticker
EEM    0
EWU    0
EZU    0
GLD    0
IEF    0
QQQ    0
SHY    0
SPY    0
VNQ    0
dtype: int64


Cellule suivante pour les risk free

In [13]:
url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=DGS3MO"
    "&cosd=2005-01-01"
)

rf = pd.read_csv(
    url,
    parse_dates=[0],
    index_col=0,
    na_values="."
)

rf.index.name = "Date"
rf.columns = ["risk_free_rate"]

print(rf.head())
print(rf.isna().sum())

rf.to_csv("../data/risk_free_rate.csv")

            risk_free_rate
Date                      
2005-01-03            2.32
2005-01-04            2.33
2005-01-05            2.33
2005-01-06            2.31
2005-01-07            2.32
risk_free_rate    232
dtype: int64


Final preparation cell, load everything

In [18]:
 
# Load ETF prices
prices = pd.read_csv(
    "../data/etf_prices_raw.csv",
    index_col="Date",
    parse_dates=True
)

# Convert prices to daily returns
returns = metrics.compute_returns(prices).dropna()

# Load annual risk-free rate
rf_rate = pd.read_csv(
    "../data/risk_free_rate.csv",
    index_col="Date",
    parse_dates=True
)["risk_free_rate"]

# Clean values
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

# Convert percent to decimal
annual_rf = rf_rate / 100

# Convert annual rate to daily return
risk_free_returns = (1 + annual_rf)**(1/252) - 1

# Match ETF trading dates
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

In [20]:
print(returns.head())
print(risk_free_returns.head())

print("ETF NaN:")
print(returns.isna().sum())

print("RF NaN:")
print(risk_free_returns.isna().sum())


                 EEM       EWU       EZU       GLD       IEF       QQQ  \
Date                                                                     
2005-01-04 -0.030788 -0.018354 -0.015217 -0.006509 -0.006226 -0.018228   
2005-01-05 -0.012242  0.000566 -0.006181 -0.001638  0.001656 -0.006188   
2005-01-06 -0.000680 -0.000566  0.002827 -0.012186  0.000944 -0.004931   
2005-01-07  0.001937  0.001133 -0.011134 -0.007355 -0.000825  0.005215   
2005-01-10  0.001253  0.000566  0.003991  0.002629 -0.000118 -0.000518   

                 SHY       SPY       VNQ  
Date                                      
2005-01-04 -0.002210 -0.012219 -0.015029  
2005-01-05  0.000247 -0.006901 -0.033243  
2005-01-06  0.000492  0.005085  0.007704  
2005-01-07  0.000123 -0.001434 -0.002237  
2005-01-10 -0.000614  0.004728 -0.003177  
Date
2005-01-04    0.000091
2005-01-05    0.000091
2005-01-06    0.000091
2005-01-07    0.000091
2005-01-10    0.000093
Name: risk_free_rate, dtype: float64
ETF NaN:
EEM    0
EWU  